In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(VGAM)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(piano)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(foreach)
    library(ggpubr)
    library(DelayedArray)
    library(rmcorr)
    library(ggh4x)
    library(hooke)
})

In [ ]:
estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
    cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
    aggregate_g1s_expression = counts(cds_g1s)
    aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
    aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)

    cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
    aggregate_g2m_expression = counts(cds_g2m)
    aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
    aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
    cds$g1s_score = log(aggregate_g1s_expression+1)
    cds$g2m_score = log(aggregate_g2m_expression+1)
    cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
    return(cds)
}

aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
    cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
    aggregate_marker_expression = counts(cds_gene_list_norm)
    aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
    aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
    aggregate_marker_expression = log(aggregate_marker_expression + 1)
    colData(cds)[[name]] = aggregate_marker_expression
    return(cds)
}

In [ ]:
cds_erythroids = readRDS("../data/cds_erythroids.RDS")

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
cc.genes$s.genes = stringr::str_to_title(cc.genes$s.genes)
cc.genes$g2m.genes = stringr::str_to_title(cc.genes$g2m.genes)

In [ ]:
cds_erythroids = estimate_cell_cycle(cds_erythroids, 
                                     g1s_markers = cc.genes$s.genes, 
                                     g2m_markers = cc.genes$g2m.genes)

In [ ]:
cds_erythroids$Grouping = paste(cds_erythroids$Experiment, cds_erythroids$Embryo, sep="-")
cds_erythroids$Cluster_group = paste0(cds_erythroids$trajectory_refined, "_", cds_erythroids$Cluster_sub)

In [ ]:
cds_erythroids = estimate_size_factors(cds_erythroids)

In [ ]:
detect_genes_fixed = function (cds, min_expr = 0) {
    assertthat::assert_that(methods::is(cds, "cell_data_set"))
    assertthat::assert_that(is.numeric(min_expr))
    mat_bin <- counts(cds) > min_expr
    rowData(cds)$num_cells_expressed <- Matrix::rowSums(mat_bin)
    colData(cds)$num_genes_expressed <- Matrix::colSums(mat_bin)
    cds
}

In [ ]:
cds_erythroids = detect_genes_fixed(cds_erythroids, 1)

In [ ]:
rowdata = as.data.frame(rowData(cds_erythroids))
genes_to_test = rowdata %>% filter(num_cells_expressed > 2000) %>% rownames()

In [ ]:
cds_erythroids_subset = cds_erythroids[genes_to_test, ]
cds_erythroids_subset

In [ ]:
pseudobulk_cds = function(cds_sub, sample_group = "Cluster_group", cell_group = "Grouping") {
    cf_ccs = new_cell_count_set(cds_sub,
                            sample_group = sample_group,
                            cell_group = cell_group)
    
    pb_cds = hooke:::pseudobulk_ccs_for_states(cf_ccs)
    
    return(pb_cds)
    
}

In [ ]:
pb_erythroids = pseudobulk_cds(cds_erythroids)

In [ ]:
metadata = as.data.frame(colData(cds_erythroids))
pb_metadata = as.data.frame(colData(pb_erythroids))

In [ ]:
head(pb_metadata, 3)

In [ ]:
pb_scores = pb_metadata %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, cell_group, sample, Pseudotime) %>%
    arrange(cell_group, sample) %>%
    select(Grouping = cell_group, Cluster_group= sample, Pseudotime)

head(pb_scores, 3)

In [ ]:
orig_scores = metadata %>%
    group_by(Experiment, Embryo, Sex, trajectory_refined, Cluster_sub, Grouping, Cluster_group) %>%
    summarise(Pseudotime = mean(Pseudotime)) %>%
    as.data.frame() %>%
    #arrange(Grouping, Cluster_group) %>%
    select(Grouping, Cluster_group, Pseudotime)

head(orig_scores, 3)

In [ ]:
options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
score_comp_df = orig_scores %>%
    left_join(., pb_scores, by = c("Grouping", "Cluster_group")) %>%
    rename("Pseudotime_orig" = "Pseudotime.x", "Pseudotime_pb" = "Pseudotime.y") 

score_comp_df %>%
    ggplot(., aes(Pseudotime_orig, Pseudotime_pb)) + 
        geom_point(size = 0.5) + 
        monocle3:::monocle_theme_opts()

In [ ]:
metadata_subset = metadata %>%
    group_by(Experiment, Embryo, Sex, trajectory_refined, Cluster_sub, Grouping, Cluster_group) %>%
    summarise(Pseudotime_avg = mean(Pseudotime)) %>%
    select(Experiment, Embryo, Sex, trajectory_refined, Cluster_sub, 
           cell_group = Grouping, sample = Cluster_group, Pseudotime_avg) 

metadata_subset = pb_metadata %>%
    select(cell_group, sample) %>%
    left_join(., metadata_subset, by = c("cell_group", "sample"))

In [ ]:
pb_erythroids$Experiment = NULL
pb_erythroids$Embryo = NULL
pb_erythroids$Sex = NULL
pb_erythroids$trajectory_refined = NULL
pb_erythroids$Cluster_sub = NULL
pb_erythroids$Pseudotime = NULL

pb_erythroids$Experiment = metadata_subset$Experiment
pb_erythroids$Embryo = metadata_subset$Embryo
pb_erythroids$Sex = metadata_subset$Sex
pb_erythroids$trajectory_refined = metadata_subset$trajectory_refined
pb_erythroids$Cluster_sub = metadata_subset$Cluster_sub
pb_erythroids$Pseudotime_avg = metadata_subset$Pseudotime_avg

In [ ]:
pb_metadata = as.data.frame(colData(pb_erythroids))

pb_scores = pb_metadata %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, cell_group, sample, Pseudotime_avg) %>%
    arrange(cell_group, sample) %>%
    select(Grouping = cell_group, Cluster_group= sample, Pseudotime_avg)

orig_scores = metadata %>%
    group_by(Experiment, Embryo, Sex, trajectory_refined, Cluster_sub, Grouping, Cluster_group) %>%
    summarise(Pseudotime_avg = mean(Pseudotime)) %>%
    as.data.frame() %>%
    select(Grouping, Cluster_group, Pseudotime_avg)

options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
score_comp_df = orig_scores %>%
    left_join(., pb_scores, by = c("Grouping", "Cluster_group")) %>%
    rename("Pseudotime_avg_orig" = "Pseudotime_avg.x", "Pseudotime_avg_pb" = "Pseudotime_avg.y") 

score_comp_df %>%
    ggplot(., aes(Pseudotime_avg_orig, Pseudotime_avg_pb)) + 
        geom_point(size = 0.5) + 
        monocle3:::monocle_theme_opts()

In [ ]:
pb_erythroids = detect_genes_fixed(pb_erythroids, 0.05)

rowdata = as.data.frame(rowData(pb_erythroids))

genes = rowdata %>%
            filter(num_cells_expressed > 0.15*ncol(pb_erythroids))
dim(genes)

## Graph test

In [ ]:
# trajectory_genes <- graph_test(cds_erythroids_subset, 
#                                neighbor_graph = "principal_graph", 
#                                cores = 10)

trajectory_genes = readRDS("../data/erythroids_graph_test_pseudotime.RDS")

In [ ]:
sig_genes_df <- trajectory_genes %>%
  filter(q_value < 0.05 & morans_I > 0.05) %>%
  arrange(desc(morans_I))
dim(sig_genes_df)

In [ ]:
sig_genes_df %>% head()

In [ ]:
model_predictions <- function(model_tbl, new_data, type="response") {
    predict_helper <- function(model, cds){
        tryCatch({
            stats::predict(model, newdata=new_data, type=type)
        }, error = function(e){
            retval = rep_len(NA, nrow(new_data))
            names(retval) = row.names(new_data)
            return(retval)
        })
    }
    model_tbl <- model_tbl %>%
    dplyr::mutate(predictions = purrr::map(model, predict_helper, new_data))
    pred_matrix = t(do.call(cbind, model_tbl$predictions))
    return(pred_matrix)
}

model_predictions <- function(model_tbl, new_data, type="response") {
  
    predict_helper <- function(mod){
        tryCatch({
          stats::predict(mod, newdata=new_data, type=type)
        }, error = function(e){
          retval = rep_len(NA, nrow(new_data))
          names(retval) = row.names(new_data)
          return(retval)
        })
    }

    predictions_list <- purrr::map(model_tbl$model, predict_helper)

    pred_matrix <- t(do.call(cbind, predictions_list))

    rownames(pred_matrix) <- model_tbl$gene_short_name 

    return(pred_matrix)
}

plot_pseudotime_heatmap <- function(cds_subset, cluster_rows = TRUE, bin_size = 100,  
                hclust_method = "ward.D2", min_expr = 0.2, ann_colors = "black",
    num_clusters = 3, hmcols = NULL, add_annotation_row = NULL, annotation_row = NA,
    add_annotation_col = NULL, show_rownames = FALSE, use_gene_short_name = TRUE, 
    norm_method = c("Size_Factor"), scale_max = 3, scale_min = -3, 
    trend_formula = "~ splines::ns(Pseudotime_avg, df=3)", main = "", return_heatmap = FALSE, 
    cores = 1, file_name = NA) 
{

    newdata <- data.frame(Pseudotime_avg = seq(min(pData(cds_subset)$Pseudotime_avg), 
                            max(pData(cds_subset)$Pseudotime_avg), length.out = bin_size),
                         Size_Factor = 1)

    cds_exprs <- SingleCellExperiment::counts(cds_subset)
    cds_exprs <- Matrix::t(Matrix::t(cds_exprs)/size_factors(cds_subset))

    
    new_data <- data.frame(Pseudotime_avg = cds_subset$Pseudotime_avg)
    model_tbl = fit_models(cds_subset, model_formula_str = trend_formula,                    
                           #expression_family = "negbinomial",
                           reduction_method="UMAP",
                           cores = cores,
                           clean_model = TRUE,
                           verbose = FALSE)

    m <- model_predictions(model_tbl, new_data = newdata)

    if (use_gene_short_name) {
        rownames(m) = rowData(cds_subset)$gene_short_name
    }
    
    m = m[!apply(m, 1, sd) == 0, ]
    m = Matrix::t(scale(Matrix::t(m), center = TRUE))
    m = m[is.na(row.names(m)) == FALSE, ]
    m[is.nan(m)] = 0
    m[m > scale_max] = scale_max
    m[m < scale_min] = scale_min
    heatmap_matrix <- m
    row_dist <- as.dist((1 - cor(Matrix::t(heatmap_matrix)))/2)
    row_dist[is.na(row_dist)] <- 1
    if (is.null(hmcols)) {
        bks <- seq(-3.1, 3.1, by = 0.1)
        hmcols <- viridis::inferno(length(bks) - 1)
    } else {
        bks <- seq(-3.1, 3.1, length.out = length(hmcols))
    }
    
    pt_labels <- rep("", ncol(heatmap_matrix))
    
    max_pt <- max(newdata$Pseudotime_avg, na.rm = TRUE)
    target_values <- seq(0, max_pt, by = 2)

    for (val in target_values) {
        closest_bin_index <- which.min(abs(newdata$Pseudotime_avg - val))
        pt_labels[closest_bin_index] <- as.character(val)
    }

    colnames(heatmap_matrix) <- pt_labels
    
    hmcols <- viridis::inferno(length(bks) - 1)
    
    ph <- pheatmap(heatmap_matrix, useRaster = T, cluster_cols = F, 
        cluster_rows = cluster_rows, show_rownames = show_rownames, show_colnames = F, 
        silent = TRUE, filename = NA, 
        breaks = bks, color = hmcols)
    
    
    if (!is.null(add_annotation_row)) {
        old_colnames_length <- ncol(annotation_row)
        annotation_row <- cbind(annotation_row, add_annotation_row[row.names(annotation_row), 
            ])
        colnames(annotation_row)[(old_colnames_length + 1):ncol(annotation_row)] <- colnames(add_annotation_row)
    }
    
    else {
        feature_label <- rownames(heatmap_matrix)
        row_ann_labels <- rownames(annotation_row)
    }
    
    
    rownames(heatmap_matrix) <- feature_label
    colnames(heatmap_matrix) <- c(1:ncol(heatmap_matrix))
    
    ph_res <- pheatmap(heatmap_matrix[, ], 
        useRaster = T, 
        cluster_cols = FALSE, 
        
        cluster_rows = cluster_rows, 
        
        show_rownames = show_rownames, 
        show_colnames = TRUE,       
        labels_col = pt_labels,      
        angle_col = "0",             
        fontsize_col = 10,           
        silent=F, border_color = NA,
        treeheight_row = 30, 
        main = main,
        breaks = bks, fontsize = 10, color = hmcols,

        annotation_row = if(all(is.na(annotation_row))) NA else annotation_row,
        
        annotation_names_row = F, annotation_names_col = F,
        fontsize_row = 12, filename = file_name, legend=F)
    grid::grid.rect(gp = grid::gpar("fill", col = NA))
    grid::grid.draw(ph_res$gtable)
    
    if (return_heatmap) {
        ph_res$annotation_row = annotation_row
        return(list(
            heatmap = ph_res,
            matrix = heatmap_matrix
        ))
    }
}

In [ ]:
options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=300)
hp = plot_pseudotime_heatmap(pb_erythroids[rownames(sig_genes_df),], return_heatmap=T)

In [ ]:
step1_result <- plot_pseudotime_heatmap(
    pb_erythroids[rownames(sig_genes_df),], 
    cluster_rows = TRUE, 
    return_heatmap = TRUE
)

my_matrix <- step1_result$matrix
gene_tree <- step1_result$heatmap$tree_row

gene_clusters <- cutree(gene_tree, k = 6)

gene_cluster_anno <- data.frame(
    Cluster = as.factor(gene_clusters) 
)
rownames(gene_cluster_anno) <- names(gene_clusters)

## Supplementary Figure 15C
final_result <- plot_pseudotime_heatmap(
    pb_erythroids[rownames(sig_genes_df),], 
    cluster_rows = TRUE, 
    annotation_row = gene_cluster_anno,
    return_heatmap = TRUE
)

In [ ]:
cluster_3_genes <- gene_cluster_anno %>%
    filter(Cluster == 3) %>%
    rownames_to_column("gene_short_name")

cluster_3_genes

In [ ]:
metadata_erythroids = as.data.frame(colData(cds_erythroids))
umap_coordinates = reducedDims(cds_erythroids)[['UMAP']]

metadata_erythroids = metadata_erythroids %>%
             mutate(UMAP1 = umap_coordinates[,1],
                    UMAP2 = umap_coordinates[,2], 
                    UMAP3 = umap_coordinates[,3])

# Load data

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30_all_genes_pseudotime_final.RDS")

In [ ]:
metadata = as.data.frame(colData(cds_VCS))

umap_coordinates = reducedDims(cds_VCS)[['UMAP']]

metadata = metadata %>%
             mutate(UMAP1 = umap_coordinates[,1],
                    UMAP2 = umap_coordinates[,2], 
                    UMAP3 = umap_coordinates[,3])

In [ ]:
cds_VCS$UMAP1 = metadata$UMAP1
cds_VCS$UMAP2 = metadata$UMAP2
cds_VCS$UMAP3 = metadata$UMAP3

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
cc.genes$s.genes = stringr::str_to_title(cc.genes$s.genes)
cc.genes$g2m.genes = stringr::str_to_title(cc.genes$g2m.genes)

In [ ]:
cds_VCS = estimate_cell_cycle(cds_VCS, g1s_markers = cc.genes$s.genes, g2m_markers = cc.genes$g2m.genes)

In [ ]:
cds_erythroids = cds_VCS[, cds_VCS$trajectory_refined == "Definitive_erythroid"]
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Cd36'), name = 'Cd36')
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Slc4a1'), name = 'Slc4a1')
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Gypa'), name = 'Gypa')
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Hbb-bs', 'Hbb-bt', 'Hbb-bh1', 'Hbb-y'), 
                                      name = 'Hemoglobin')
metadata_erythroids = as.data.frame(colData(cds_erythroids))

In [ ]:
umap_coordinates = reducedDims(cds_erythroids)[['UMAP']]

metadata_erythroids = metadata_erythroids %>%
             mutate(UMAP1 = umap_coordinates[,1],
                    UMAP2 = umap_coordinates[,2], 
                    UMAP3 = umap_coordinates[,3])

In [ ]:
min_max_scale <- function(x) {
  (x - min(x, na.rm = TRUE)) / (max(x, na.rm = TRUE) - min(x, na.rm = TRUE))
}

get_loess_normalized <- function(df, x_col, y_col) {
  clean_df <- df[!is.na(df[[x_col]]) & !is.na(df[[y_col]]), ]
  
  fit <- loess(as.formula(paste(y_col, "~", x_col)), data = clean_df, span = 0.9)

  res <- data.frame(x = seq(min(clean_df[[x_col]]), max(clean_df[[x_col]]), length.out = 100))
  colnames(res)[1] <- x_col
  pred <- predict(fit, newdata = res, se = TRUE)
  
  y_fit <- pred$fit
  y_upper <- pred$fit + (1.96 * pred$se.fit)
  y_lower <- pred$fit - (1.96 * pred$se.fit)
  
  mn <- min(y_fit, na.rm = TRUE)
  mx <- max(y_fit, na.rm = TRUE)
  
  res$y_n   <- (y_fit - mn) / (mx - mn)
  res$y_u_n <- (y_upper - mn) / (mx - mn)
  res$y_l_n <- (y_lower - mn) / (mx - mn)
  
  return(res)
}

In [ ]:
base_data <- metadata_erythroids %>%
    group_by(Experiment, trajectory_refined, Embryo, Cluster_sub) %>%
    summarise(
        med_prolif = median(proliferation_index, na.rm = TRUE),
        pt_all = median(Pseudotime_avg, na.rm = TRUE),
        med_score = median(avg_score[Marker == "Nucleolin" & Phenotype == "Ratio"], na.rm = TRUE),
        pt_filtered = median(Pseudotime_avg[Marker == "Nucleolin" & Phenotype == "Ratio"], na.rm = TRUE),
        .groups = "drop"
    ) %>%
    filter(!is.na(med_score) & !is.na(med_prolif)) %>%
    mutate(
        norm_prolif = min_max_scale(med_prolif),
        norm_score = min_max_scale(med_score)
    )

pts_score <- base_data %>%
    group_by(pt_filtered) %>%
    summarise(col_score = median(norm_score, na.rm = TRUE), .groups = "drop") %>%
    mutate(col_score_norm = min_max_scale(col_score))

pts_prolif <- base_data %>%
    group_by(pt_all) %>%
    summarise(col_prolif = median(norm_prolif, na.rm = TRUE), .groups = "drop") %>%
    mutate(col_prolif_norm = min_max_scale(col_prolif))

l1_score <- get_loess_normalized(base_data, "pt_filtered", "norm_score")
l1_prolif <- get_loess_normalized(base_data, "pt_all", "norm_prolif")

## Figure 4C
options(repr.plot.width=3.5, repr.plot.height=3, repr.plot.res=300)
ggplot() + 
    geom_ribbon(data = l1_score, aes(x = pt_filtered, ymin = y_l_n, ymax = y_u_n), fill = "grey70", alpha = 0.4) + 
    geom_ribbon(data = l1_prolif, aes(x = pt_all, ymin = y_l_n, ymax = y_u_n), fill = "#fb9a99", alpha = 0.25) + 
    geom_line(data = l1_score, aes(x = pt_filtered, y = y_n), linewidth = 1.2, color = "black") + 
    geom_line(data = l1_prolif, aes(x = pt_all, y = y_n), linewidth = 1.2, color = "#fb9a99") + 
    geom_point(data = pts_score, aes(x = pt_filtered, y = col_score_norm), size = 1.5, shape = 3, color = "black") + 
    geom_point(data = pts_prolif, aes(x = pt_all, y = col_prolif_norm), size = 1.5, shape = 17, color = "black") + 
    labs(x = "Pseudotime", y = "Normalized score") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "none", 
          axis.text = element_text(color="black", size = 11), 
          axis.ticks = element_line(color = "black"), 
          aspect.ratio = 1)

In [ ]:
plot_phenotype_proliferation <- function(metadata, target_phenotype) {
    
    get_marker_df <- function(marker_name, is_prolif = FALSE) {
        if (is_prolif) {
            metadata %>%
                group_by(Experiment, trajectory_refined, Embryo, Cluster_sub) %>%
                summarise(val = median(proliferation_index, na.rm = TRUE),
                          pt  = median(Pseudotime_avg, na.rm = TRUE), .groups = "drop") %>%
                mutate(norm_val = min_max_scale(val))
        } else {
            metadata %>%
                filter(Marker == marker_name & Phenotype == target_phenotype) %>%
                group_by(Experiment, trajectory_refined, Embryo, Cluster_sub) %>%
                summarise(val = median(avg_score, na.rm = TRUE),
                          pt  = median(Pseudotime_avg, na.rm = TRUE), .groups = "drop") %>%
                mutate(norm_val = min_max_scale(val))
        }
    }
    
    l_pro  <- get_loess_normalized(get_marker_df(is_prolif = TRUE), "pt", "norm_val")
    l_nuc  <- get_loess_normalized(get_marker_df("Nucleolin"), "pt", "norm_val")
    l_sc35 <- get_loess_normalized(get_marker_df("SC35"), "pt", "norm_val")
    
    peak_pt_prolif <- l_pro$pt[which.max(l_pro$y_n)]
    
    peak_y_nuc <- max(l_nuc$y_n, na.rm = TRUE)
    peak_y_sc35 <- max(l_sc35$y_n, na.rm = TRUE)
    
    y_nuc_at_prolif_peak <- l_nuc$y_n[which.min(abs(l_nuc$pt - peak_pt_prolif))]
    y_sc35_at_prolif_peak <- l_sc35$y_n[which.min(abs(l_sc35$pt - peak_pt_prolif))]
    
    drop_nuc <- peak_y_nuc - y_nuc_at_prolif_peak
    drop_sc35 <- peak_y_sc35 - y_sc35_at_prolif_peak
    
    cat("\n=== STATISTICS FOR", target_phenotype, "===\n")
    cat("Proliferation peak (pseudotime):", round(peak_pt_prolif, 3), "\n")
    cat("Nucleolin drop from its peak to prolif peak:", round(drop_nuc, 4), "\n")
    cat("SC35 drop from its peak to prolif peak:", round(drop_sc35, 4), "\n")
    cat("===================================\n\n")

    p <- ggplot() +
        geom_ribbon(data = l_pro,  aes(x=pt, ymin=y_l_n, ymax=y_u_n, fill="Proliferation"), alpha=0.15) +
        geom_ribbon(data = l_nuc,  aes(x=pt, ymin=y_l_n, ymax=y_u_n, fill="Nucleolin"), alpha=0.1) +
        geom_ribbon(data = l_sc35, aes(x=pt, ymin=y_l_n, ymax=y_u_n, fill="SC35"), alpha=0.1) +
        
        geom_line(data = l_pro,  aes(x=pt, y=y_n, color="Proliferation"), linewidth=1.3) +
        geom_line(data = l_nuc,  aes(x=pt, y=y_n, color="Nucleolin"), linewidth=1.3) +
        geom_line(data = l_sc35, aes(x=pt, y=y_n, color="SC35"), linewidth=1.3) +
        
        scale_color_manual(name = "", 
                           values = c("Proliferation" = "#fb9a99", "Nucleolin" = "black", "SC35" = "#1f78b4")) +
        scale_fill_manual(name = "", 
                          values = c("Proliferation" = "#fb9a99", "Nucleolin" = "black", "SC35" = "#1f78b4")) +
        labs(x = "Pseudotime", y = "Normalized score") +
        monocle3:::monocle_theme_opts() +
        guides(color = guide_legend(override.aes = list(alpha = 1, size = 1.5)), fill = "none") + 
        theme(aspect.ratio = 1,
              legend.position = "top", axis.title = element_text(color = "black", size = 12),
              strip.text = element_text(color = "black", size = 11, face = "bold"),
              axis.text = element_text(color = "black", size = 11),
              legend.text = element_text(color = "black", size = 10),
              axis.ticks = element_line(color = "black"))
              
    return(p)
}

## Supplementary 15A,B
options(repr.plot.width=3.5, repr.plot.height=4, repr.plot.res=300)
plot_phenotype_proliferation(metadata_erythroids, "Ratio")

plot_phenotype_proliferation(metadata_erythroids, "Area")

In [ ]:
peak_x_score <- loess_score_data$med_Pseudotime_filtered[which.max(loess_score_data$y_norm)]
peak_x_prolif <- loess_prolif_data$med_Pseudotime_all[which.max(loess_prolif_data$y_norm)]

print(paste("Peak Nucleolin Pseudotime:", round(peak_x_score, 3)))
print(paste("Peak Proliferation Pseudotime:", round(peak_x_prolif, 3)))

In [ ]:
data.frame(gene_correlations) %>%
    arrange(-gene_correlations)

In [ ]:
heatmap_mat <- final_result$matrix
pheno_scores <- loess_score_data$y_norm

gene_correlations <- apply(heatmap_mat, 1, function(x) cor(x, pheno_scores, method = "pearson"))

top_genes <- names(sort(gene_correlations, decreasing = TRUE))[1:6]

                           df_plot <- as.data.frame(t(heatmap_mat[top_genes, , drop = FALSE]))
df_plot_norm <- as.data.frame(lapply(df_plot, normalize))
                           
df_plot_norm$Phenotype_Score <- pheno_scores
df_plot_norm$Pseudotime_Bin <- 1:nrow(df_plot)

df_long <- df_plot_norm %>%
    pivot_longer(cols = all_of(top_genes), 
                 names_to = "Gene", 
                 values_to = "Expression")

In [ ]:
## Supplementary 15D

options(repr.plot.width=6, repr.plot.height=9, repr.plot.res=300)

ggplot(df_long, aes(x = Pseudotime_Bin)) + 
    # Plot the Gene Expression curve
    geom_line(aes(y = Expression, color = "Gene Expression"), linewidth = 1.2) + 
    
    geom_line(aes(y = Phenotype_Score * max(Expression), color = "Nucleolar fraction"), 
              linewidth = 2, alpha = 0.25) + 
    
    facet_wrap(~ Gene, scales = "free_y", ncol=2) + 
    
    # Custom colors
    scale_color_manual(values = c("Gene Expression" = "#3182bd", "Nucleolar fraction" = "black")) +
    
    labs(x = "Pseudotime", y = "Normalized score", color = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "top", axis.title = element_text(color = "black", size = 16),
          strip.text = element_text(color = "black", size = 16, face = "bold"),
          axis.text = element_text(color = "black", size = 14),
          legend.text = element_text(color = "black", size = 13),
          axis.ticks = element_line(color = "black"))

In [ ]:
## Figure 4D
options(repr.plot.width=2.5, repr.plot.height=3, repr.plot.res=300)

ggplot(df_long %>%
           filter(Gene == "Bcl11a"), aes(x = Pseudotime_Bin)) + 

    geom_line(aes(y = Phenotype_Score * max(Expression), color = "Phenotype Score"), 
              linewidth = 2, alpha = 0.25) + 
    geom_line(aes(y = Expression, color = "Gene Expression"), linewidth = 1) + 
    
    facet_wrap(~ Gene, scales = "free_y") + 

    scale_color_manual(values = c("Gene Expression" = "#3182bd", "Phenotype Score" = "black")) +
    
    labs(x = "Pseudotime", y = "Normalized value", color = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "position",
          strip.text = element_text(color = "black", size = 12, face = "bold"),
          axis.text = element_text(color = "black", size = 12),
          axis.ticks = element_line(color = "black"))

options(repr.plot.width=2.5, repr.plot.height=3, repr.plot.res=300)

ggplot(df_long %>%
           filter(Gene == "Lin28b"), aes(x = Pseudotime_Bin)) + 

    geom_line(aes(y = Phenotype_Score * max(Expression), color = "Phenotype Score"), 
              linewidth = 2, alpha = 0.25) + 
    geom_line(aes(y = Expression, color = "Gene Expression"), linewidth = 1) + 
    
    facet_wrap(~ Gene, scales = "free_y") + 
    
    scale_color_manual(values = c("Gene Expression" = "#3182bd", "Phenotype Score" = "black")) +
    
    labs(x = "Pseudotime", y = "Normalized value", color = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "position",
          strip.text = element_text(color = "black", size = 12, face = "bold"),
          axis.text = element_text(color = "black", size = 12),
          axis.ticks = element_line(color = "black"))

In [ ]:
heatmap_mat <- final_result$matrix
pheno_scores <- loess_score_data$y_norm

target_peak_bin <- which.max(pheno_scores)
print(paste("Nucleolar score peaks at bin:", target_peak_bin))

gene_peak_bins <- apply(heatmap_mat, 1, which.max)
window_size <- 5

orthogonal_genes <- names(gene_peak_bins[abs(gene_peak_bins - target_peak_bin) <= window_size])
print(paste(length(orthogonal_genes), "genes peaks around the same range as the nucleolar score"))

In [ ]:
high_cor_genes <- names(gene_correlations[gene_correlations > 0.98])

candidates <- intersect(orthogonal_genes, high_cor_genes)

candidates

In [ ]:
data.frame(pearson = gene_correlations) %>%
    arrange(-pearson) %>%
    rownames_to_column("gene_short_name") %>%
    filter(gene_short_name %in% orthogonal_genes) %>%
    filter(pearson > 0.95) %>%
    filter(gene_short_name %in% c("Bcl11a", "Lin28b"))